### DATA INGESTION

In [137]:
from langchain_core.documents import Document

In [138]:
doc = Document(
    page_content="This is a main text document i am using to create RAG.",
    metadata={"source": "my_file.txt",
              "author": "Pawan Balpande",
              "date":"2024-06-01",
              "pages": 10}
)
doc


Document(metadata={'source': 'my_file.txt', 'author': 'Pawan Balpande', 'date': '2024-06-01', 'pages': 10}, page_content='This is a main text document i am using to create RAG.')

In [139]:
### Simple TXT file
import os
os.makedirs("data/textfile", exist_ok=True)

In [140]:
### Text loader
from langchain_community.document_loaders import TextLoader

### Read the text file

loader = TextLoader("../data/textfile/python_intro.txt", encoding="utf-8")
loader.load()

[Document(metadata={'source': '../data/textfile/python_intro.txt'}, page_content='Python Introduction\n\nPython is a high-level, interpreted programming language known for its clean syntax and readability. It is widely used in web development, data science, automation, artificial intelligence, scripting, and software development.\n\nKey Features of Python\n- Easy to learn and read due to simple syntax.\n- Interpreted language, so code runs without separate compilation.\n- Cross-platform support (Windows, macOS, Linux).\n- Large standard library with built-in modules.\n- Supports multiple programming paradigms: procedural, object-oriented, and functional.\n- Strong community support and vast ecosystem of third-party packages.\n\nOne-Liner Info\nPython is a beginner-friendly yet powerful language used for everything from automation scripts to advanced AI systems.\n')]

In [124]:
### Diretory loader
from langchain_community.document_loaders import DirectoryLoader

### Load all the text files in the directory
dir_loader = DirectoryLoader("../data/textfile", 
                             glob="**/*.txt",## Pattern to match files in the directory
                             loader_cls=TextLoader,## load class to use for loading the files
                             loader_kwargs={"encoding": "utf-8"},
                             show_progress=False)

dir_loader.load()

[Document(metadata={'source': '../data/textfile/python_intro.txt'}, page_content='Python Introduction\n\nPython is a high-level, interpreted programming language known for its clean syntax and readability. It is widely used in web development, data science, automation, artificial intelligence, scripting, and software development.\n\nKey Features of Python\n- Easy to learn and read due to simple syntax.\n- Interpreted language, so code runs without separate compilation.\n- Cross-platform support (Windows, macOS, Linux).\n- Large standard library with built-in modules.\n- Supports multiple programming paradigms: procedural, object-oriented, and functional.\n- Strong community support and vast ecosystem of third-party packages.\n\nOne-Liner Info\nPython is a beginner-friendly yet powerful language used for everything from automation scripts to advanced AI systems.\n'),
 Document(metadata={'source': '../data/textfile/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine 

In [141]:
### Loading PDF files
from langchain_community.document_loaders import PyMuPDFLoader , PyPDFLoader

### Load all the text files in the directory
dir_loader = DirectoryLoader("../data/pdf", 
                             glob="**/*.pdf",## Pattern to match files in the directory
                             loader_cls=PyMuPDFLoader,## load class to use for loading the files
                             show_progress=False)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Codex Minimal PDF Engine', 'creator': 'Codex PDF Generator', 'creationdate': '2026-05-12T14:16:59+00:00', 'source': '../data/pdf/objectdetection.pdf', 'file_path': '../data/pdf/objectdetection.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'Object Detection Notes', 'author': 'Pawan Balpande', 'subject': 'Foundations and workflows for object detection systems', 'keywords': 'object detection,computer vision,yolo,rcnn,bounding boxes', 'moddate': '2026-05-12T14:16:59+00:00', 'trapped': '', 'modDate': "D:20260512141659+00'00'", 'creationDate': "D:20260512141659+00'00'", 'page': 0}, page_content='Object Detection Notes\nAuthor: Pawan Balpande\nCreator: Codex PDF Generator\nProducer: Codex Minimal PDF Engine\nDetection Basics\nObject detection is an important area in modern AI systems. In practical applications,\nteams combine theory with experimentation, evaluate model behavior, and iterate based on\nmeasurable outcomes. Clear documentation and consist

In [142]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [ ]:
# Creating Data Chunks 

In [143]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
    

In [144]:
chunks = split_documents(pdf_documents)
chunks

Split 24 documents into 104 chunks

Example chunk:
Content: Object Detection Notes
Author: Pawan Balpande
Creator: Codex PDF Generator
Producer: Codex Minimal PDF Engine
Detection Basics
Object detection is an important area in modern AI systems. In practical ...
Metadata: {'producer': 'Codex Minimal PDF Engine', 'creator': 'Codex PDF Generator', 'creationdate': '2026-05-12T14:16:59+00:00', 'source': '../data/pdf/objectdetection.pdf', 'file_path': '../data/pdf/objectdetection.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'Object Detection Notes', 'author': 'Pawan Balpande', 'subject': 'Foundations and workflows for object detection systems', 'keywords': 'object detection,computer vision,yolo,rcnn,bounding boxes', 'moddate': '2026-05-12T14:16:59+00:00', 'trapped': '', 'modDate': "D:20260512141659+00'00'", 'creationDate': "D:20260512141659+00'00'", 'page': 0}


[Document(metadata={'producer': 'Codex Minimal PDF Engine', 'creator': 'Codex PDF Generator', 'creationdate': '2026-05-12T14:16:59+00:00', 'source': '../data/pdf/objectdetection.pdf', 'file_path': '../data/pdf/objectdetection.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'Object Detection Notes', 'author': 'Pawan Balpande', 'subject': 'Foundations and workflows for object detection systems', 'keywords': 'object detection,computer vision,yolo,rcnn,bounding boxes', 'moddate': '2026-05-12T14:16:59+00:00', 'trapped': '', 'modDate': "D:20260512141659+00'00'", 'creationDate': "D:20260512141659+00'00'", 'page': 0}, page_content='Object Detection Notes\nAuthor: Pawan Balpande\nCreator: Codex PDF Generator\nProducer: Codex Minimal PDF Engine\nDetection Basics\nObject detection is an important area in modern AI systems. In practical applications,\nteams combine theory with experimentation, evaluate model behavior, and iterate based on\nmeasurable outcomes. Clear documentation and consist

### EMBEDDING AND VECTORSTOREDB

In [145]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List , Dict , Any , Optional , Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [146]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List , Dict , Any , Optional , Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [147]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: { self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
        
    def generate_embedding(self, text: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model not loaded.")
        print(f"Generating embeddings for {len(text)} texts.")
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("Embeddings generated successfully. Shape:", embeddings.shape)
        return embeddings


embedding_manager = EmbeddingManager()
embedding_manager 

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4589.39it/s]


Model loaded successfully. Embedding dimension: 384


### VECTOR STORE DB

In [148]:
class VectorStore:
    """ Manages document embeddings in a vector database (ChromaDB) """
    
    def __init__(self , collection_name: str = "pdf_documents", persist_directory:str ="../data/vectorstore" ):
        
        """ Initializes the vector store with collection name and persistence directory. """
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """ Initializes the ChromaDB client and collection. """
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name , 
                metadata = {"description": "PDF document embeddings for RAG"}
            )
            print(f"vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    
    def add_documents(self , documents: List[Any] , embeddings : np.ndarray):
        """ Adds documents to the vector store after generating embeddings. """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store.")
        
        ids = []
        metadatas = []
        document_texts = []
        embeddings_list = []
        
        for i , (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata ["doc_index"] = i 
            metadata ["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            
            ### Document content
            document_texts.append(doc.page_content)
            
            ### Embeddings
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore
            

vector store initialized. Collection: pdf_documents
Existing documents in collection: 212


In [149]:
chunks


[Document(metadata={'producer': 'Codex Minimal PDF Engine', 'creator': 'Codex PDF Generator', 'creationdate': '2026-05-12T14:16:59+00:00', 'source': '../data/pdf/objectdetection.pdf', 'file_path': '../data/pdf/objectdetection.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'Object Detection Notes', 'author': 'Pawan Balpande', 'subject': 'Foundations and workflows for object detection systems', 'keywords': 'object detection,computer vision,yolo,rcnn,bounding boxes', 'moddate': '2026-05-12T14:16:59+00:00', 'trapped': '', 'modDate': "D:20260512141659+00'00'", 'creationDate': "D:20260512141659+00'00'", 'page': 0}, page_content='Object Detection Notes\nAuthor: Pawan Balpande\nCreator: Codex PDF Generator\nProducer: Codex Minimal PDF Engine\nDetection Basics\nObject detection is an important area in modern AI systems. In practical applications,\nteams combine theory with experimentation, evaluate model behavior, and iterate based on\nmeasurable outcomes. Clear documentation and consist

In [150]:
### CONVERT THE TEXT TO EMBEDDINGS
texts= [doc.page_content for doc in chunks]

### GENERATE EMBEDDINGS

embeddings = embedding_manager.generate_embedding(texts)

### Store in vector database
vectorstore.add_documents(chunks , embeddings)

Generating embeddings for 104 texts.


Batches: 100%|██████████| 4/4 [00:01<00:00,  3.83it/s]

Embeddings generated successfully. Shape: (104, 384)
Adding 104 documents to vector store.
Successfully added 104 documents to vector store.
Total documents in collection after addition: 316


### RETRIVAL PIPELINE FROM VECTORSTORE

In [151]:
class RagRetriever:
    """ Retrieves relevant documents from the vector store based on query similarity. """
    def __init__(self, vector_store : VectorStore , embedding_manager : EmbeddingManager):
        """
        Initializes the retrival 
        
        ARGS:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager to generate query embeddings
        """
        
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query:  str , top_k: int = 5 , score_threshold: float = 0.0) -> List[Dict[str,Any]]:
        """
        Retrieves relevant documents based on query similarity.
        
        Args:
            query: User query string
            top_k: Number of top similar results to return
            score_threshold: Minimum similarity score for retrieval
        
        Returns:
            List of dictionaries containing retrieved documents and their metadata
        """
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        
        # Search the vector store for similar documents
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:  # Check if there are any results
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id , documents , metadata , distance) in enumerate(zip(ids , documents , metadatas , distances)):
                    similarity_score = 1 - distance  # Convert distance to similarity
                
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": documents,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                        
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                 print("No documents retrieved.")
                 
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RagRetriever(vectorstore , embedding_manager)
rag_retriever

In [152]:
rag_retriever.retrieve("Why attention is important?")

Retrieving documents for query: 'Why attention is important?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.63it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_af1f58ab_2',
  'content': 'Attention in AI',
  'metadata': {'content_length': 15,
   'file_path': '../data/pdf/attention.pdf',
   'title': 'Attention in AI',
   'keywords': 'attention,transformer,nlp,deep learning',
   'doc_index': 2,
   'subject': 'Overview of attention mechanism in machine learning',
   'page': 0,
   'format': 'PDF 1.4',
   'creationdate': '2026-05-12T14:03:00+00:00',
   'total_pages': 1,
   'author': 'Pawan Balpande',
   'moddate': '2026-05-12T14:03:00+00:00',
   'modDate': "D:20260512140300+00'00'",
   'trapped': '',
   'producer': 'Codex Minimal PDF Engine',
   'creationDate': "D:20260512140300+00'00'",
   'source': '../data/pdf/attention.pdf',
   'creator': 'Codex PDF Generator'},
  'similarity_score': 0.3049464225769043,
  'distance': 0.6950535774230957,
  'rank': 1},
 {'id': 'doc_e0edc4e4_56',
  'content': 'measurable outcomes. Clear documentation and consistent terminology improve collaboration\nacross teams. Self-attention computes relationships 

In [106]:
rag_retriever.retrieve("What is Embeddings and why it is important for RAG?")

Retrieving documents for query: 'What is Embeddings and why it is important for RAG?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_28f9a513_1',
  'content': 'Embeddings Basics',
  'metadata': {'creationDate': "D:20260512140300+00'00'",
   'doc_index': 1,
   'total_pages': 1,
   'author': 'Pawan Balpande',
   'producer': 'Codex Minimal PDF Engine',
   'creationdate': '2026-05-12T14:03:00+00:00',
   'format': 'PDF 1.4',
   'page': 0,
   'title': 'Embeddings Basics',
   'creator': 'Codex PDF Generator',
   'subject': 'Introduction to embeddings and vector representations',
   'keywords': 'embeddings,vectors,semantic search,rag',
   'file_path': '../data/pdf/embeddings.pdf',
   'source': '../data/pdf/embeddings.pdf',
   'modDate': "D:20260512140300+00'00'",
   'trapped': '',
   'moddate': '2026-05-12T14:03:00+00:00',
   'content_length': 17},
  'similarity_score': 0.14016014337539673,
  'distance': 0.8598398566246033,
  'rank': 1}]

In [107]:
rag_retriever.retrieve("Why Object detection is important?")

Retrieving documents for query: 'Why Object detection is important?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_1ef4e916_0',
  'content': 'Object Detection Overview',
  'metadata': {'source': '../data/pdf/objectdetection.pdf',
   'trapped': '',
   'page': 0,
   'format': 'PDF 1.4',
   'creator': 'Codex PDF Generator',
   'content_length': 25,
   'doc_index': 0,
   'creationDate': "D:20260512140300+00'00'",
   'total_pages': 1,
   'producer': 'Codex Minimal PDF Engine',
   'file_path': '../data/pdf/objectdetection.pdf',
   'subject': 'Basics of object detection in computer vision',
   'creationdate': '2026-05-12T14:03:00+00:00',
   'moddate': '2026-05-12T14:03:00+00:00',
   'modDate': "D:20260512140300+00'00'",
   'author': 'Pawan Balpande',
   'title': 'Object Detection Overview',
   'keywords': 'object detection,computer vision,yolo,bounding box'},
  'similarity_score': 0.21693694591522217,
  'distance': 0.7830630540847778,
  'rank': 1}]

In [108]:
rag_retriever.retrieve("What is RAG?")


Retrieving documents for query: 'What is RAG?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.76s/it]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [134]:
rag_retriever.retrieve("What is RAG ? , Why it is matters and How the rag process works?")

Retrieving documents for query: 'What is RAG ? , Why it is matters and How the rag process works?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.62it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [136]:
rag_retriever.retrieve("What is Proposal")

Retrieving documents for query: 'What is Proposal'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_2cf08805_3',
  'content': 'Project Proposal Guide',
  'metadata': {'source': '../data/pdf/proposal.pdf',
   'trapped': '',
   'subject': 'How to structure a technical project proposal',
   'page': 0,
   'keywords': 'proposal,planning,timeline,budget,scope',
   'producer': 'Codex Minimal PDF Engine',
   'format': 'PDF 1.4',
   'content_length': 22,
   'creationDate': "D:20260512140300+00'00'",
   'total_pages': 1,
   'modDate': "D:20260512140300+00'00'",
   'file_path': '../data/pdf/proposal.pdf',
   'doc_index': 3,
   'moddate': '2026-05-12T14:03:00+00:00',
   'creator': 'Codex PDF Generator',
   'creationdate': '2026-05-12T14:03:00+00:00',
   'author': 'Pawan Balpande',
   'title': 'Project Proposal Guide'},
  'similarity_score': 0.2624130845069885,
  'distance': 0.7375869154930115,
  'rank': 1},
 {'id': 'doc_05e072cc_77',
  'content': 'Technical Proposal Playbook\nAuthor: Pawan Balpande\nCreator: Codex PDF Generator\nProducer: Codex Minimal PDF Engine\nProposal Structure

In [154]:
rag_retriever.retrieve("What is RAG")

Retrieving documents for query: 'What is RAG'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.63it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### INTEGRATION VECTORDB CONTEXT PIPELINE WITH LLM OUTPUT

In [159]:
### Simple Rag pipline with groq LLM
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment. Add it to .env")

llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

## Simple RAG Function
def rag_simple(query , retriever , llm , top_k=3):
   ## Retrieve relevant context
    results = retriever.retrieve(query , top_k=top_k) 
    context = "\n\n".join([doc['content'] for doc in results]) if results else "No rsponse"

ModuleNotFoundError: No module named 'langchain_groq'